# Phase 03 — Psychological Capital → Employability

## Step 12 — ElasticNet Recommendation Revalidation

මෙම notebook එකේ අරමුණ Step 11 එකේ freeze කරන ලද final ElasticNet prediction model සහ evidence-grounded recommendation system එක deployment-style revalidation process එකක් මගින් නැවත පරීක්ෂා කිරීමයි.

මෙම step එකේ model එක නැවත train, tune හෝ refit නොකරයි.

Revalidation process එකෙන් පහත කරුණු පරීක්ෂා කරයි:

- final freeze manifest integrity
- frozen model සහ related artifact hashes
- feature schema සහ feature order
- saved reference predictions reproducibility
- deterministic prediction behaviour
- low, middle සහ high boundary profiles
- invalid input rejection
- recommendation-system compatibility
- recommendation determinism
- recommendation traceability
- Phase 02-style user-facing output structure
- final performance evidence consistency
- deployment readiness

මෙම step එක successfully complete වූ පසු Phase 03 final system එක `DEPLOYMENT_READY` ලෙස classify කරයි.

In [1]:
from pathlib import Path

import json
import hashlib

import numpy as np
import pandas as pd
import joblib
import shap

d:\internship-employability-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "phase3_psycap":
    PROJECT_DIR = PROJECT_DIR.parent.parent

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "phase3_psycap_prepared.csv"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "phase3"
)

MODEL_PATH = (
    MODEL_DIR
    / "phase3_psycap_selected_model.pkl"
)

METADATA_PATH = (
    MODEL_DIR
    / "phase3_psycap_model_metadata.json"
)

SCHEMA_PATH = (
    MODEL_DIR
    / "phase3_psycap_feature_schema.json"
)

REFERENCE_PATH = (
    MODEL_DIR
    / "phase3_psycap_reference_predictions.csv"
)

PARAMS_PATH = (
    MODEL_DIR
    / "phase3_tuned_hyperparameters.json"
)

FREEZE_MANIFEST_PATH = (
    MODEL_DIR
    / "phase3_psycap_freeze_manifest.json"
)

FINAL_STATUS_PATH = (
    MODEL_DIR
    / "phase3_psycap_final_status.json"
)

STEP10_REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "recommendations"
)

STEP11_REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "final_validation"
)

REPORT_DIR = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "final_revalidation"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("PROJECT_DIR:", PROJECT_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("REPORT_DIR:", REPORT_DIR)

PROJECT_DIR: d:\internship-employability-project
MODEL_PATH: d:\internship-employability-project\models\phase3\phase3_psycap_selected_model.pkl
REPORT_DIR: d:\internship-employability-project\reports\phase3\final_revalidation


## Frozen Artifact Availability Check

Final revalidation එක ආරම්භ කිරීමට පෙර frozen model සහ related metadata, schema, reference predictions, tuned hyperparameters, freeze manifest සහ final-status files available ද යන්න verify කරයි.

In [3]:
required_artifacts = {

    "Frozen_Model":
        MODEL_PATH,

    "Model_Metadata":
        METADATA_PATH,

    "Feature_Schema":
        SCHEMA_PATH,

    "Reference_Predictions":
        REFERENCE_PATH,

    "Tuned_Hyperparameters":
        PARAMS_PATH,

    "Freeze_Manifest":
        FREEZE_MANIFEST_PATH,

    "Final_Status":
        FINAL_STATUS_PATH
}


artifact_availability_rows = []

for name, path in required_artifacts.items():

    artifact_availability_rows.append({

        "Artifact":
            name,

        "Path":
            str(path),

        "Exists":
            path.exists()
    })


artifact_availability_df = pd.DataFrame(
    artifact_availability_rows
)

artifact_availability_df

,Artifact,Path,Exists
0,Frozen_Model,d:\internship-employability-project\models\pha...,True
1,Model_Metadata,d:\internship-employability-project\models\pha...,True
2,Feature_Schema,d:\internship-employability-project\models\pha...,True
3,Reference_Predictions,d:\internship-employability-project\models\pha...,True
4,Tuned_Hyperparameters,d:\internship-employability-project\models\pha...,True
5,Freeze_Manifest,d:\internship-employability-project\models\pha...,True
6,Final_Status,d:\internship-employability-project\models\pha...,True


Stop if something is missing

In [4]:
if not artifact_availability_df["Exists"].all():

    missing = artifact_availability_df.loc[
        ~artifact_availability_df["Exists"],
        "Artifact"
    ].tolist()

    raise FileNotFoundError(
        f"Missing frozen artifacts: {missing}"
    )

print(
    "All frozen artifacts are available."
)

All frozen artifacts are available.


Load freeze manifest and verify frozen status

In [5]:
with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:

    freeze_manifest = json.load(
        file
    )

print(
    "Model status:",
    freeze_manifest[
        "model_status"
    ]
)

print(
    "Retraining allowed:",
    freeze_manifest[
        "retraining_allowed"
    ]
)

Model status: FROZEN
Retraining allowed: False


Verify Step 11 final status

In [6]:
with open(
    FINAL_STATUS_PATH,
    "r",
    encoding="utf-8"
) as file:

    final_status_record = json.load(
        file
    )

print(
    "Step 11 final status:",
    final_status_record[
        "status"
    ]
)

Step 11 final status: FROZEN_AND_VALIDATED


Cryptographic artifact revalidation

In [7]:
def calculate_sha256(
    file_path
):

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        for block in iter(
            lambda: file.read(
                4096
            ),
            b""
        ):

            sha256.update(
                block
            )

    return sha256.hexdigest()

Verify each manifest artifact

In [8]:
artifact_hash_validation_rows = []

for artifact_name, artifact_info in (
    freeze_manifest[
        "artifacts"
    ].items()
):

    artifact_path = Path(
        artifact_info[
            "path"
        ]
    )

    expected_hash = (
        artifact_info[
            "sha256"
        ]
    )

    current_hash = calculate_sha256(
        artifact_path
    )

    artifact_hash_validation_rows.append({

        "Artifact":
            artifact_name,

        "Expected_SHA256":
            expected_hash,

        "Current_SHA256":
            current_hash,

        "Hash_Match":
            current_hash
            ==
            expected_hash
    })


artifact_hash_validation_df = (
    pd.DataFrame(
        artifact_hash_validation_rows
    )
)

artifact_hash_validation_df

,Artifact,Expected_SHA256,Current_SHA256,Hash_Match
0,model,6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b88...,6d8ed0af743a35f91dccf7f85630056d5a9afb551f2b88...,True
1,metadata,6004ce747d809df9eabbb34155c883c823b0515e34d154...,6004ce747d809df9eabbb34155c883c823b0515e34d154...,True
2,feature_schema,4668d129b04d180e33e566130d437de3075fbd2cd96b7f...,4668d129b04d180e33e566130d437de3075fbd2cd96b7f...,True
3,reference_predictions,4709e6eeb259f19c4d4ff287e98c38b73a145cc7da71d2...,4709e6eeb259f19c4d4ff287e98c38b73a145cc7da71d2...,True
4,tuned_hyperparameters,9280199096ff1db39bf2e837e2d4e9f324d5b52575081b...,9280199096ff1db39bf2e837e2d4e9f324d5b52575081b...,True


Load exact frozen model

In [9]:
final_model = joblib.load(
    MODEL_PATH
)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    metadata = json.load(
        file
    )

with open(
    SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as file:

    feature_schema = json.load(
        file
    )

print(
    "Frozen model:",
    metadata[
        "model_name"
    ]
)

print(
    final_model
)

Frozen model: ElasticNet
Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 ElasticNet(alpha=np.float64(0.14873521072935117), l1_ratio=0.0,
                            max_iter=50000, random_state=42))])


Verify expected structure

In [10]:
EXPECTED_FEATURES = [
    "Hope_avg",
    "Optimism_avg",
    "Resilience_avg",
    "SelfEfficacy_avg"
]

EXPECTED_TARGET = "EVS_avg"

pipeline_steps = list(
    final_model.named_steps.keys()
)

scaler_name = (
    final_model
    .named_steps[
        "scaler"
    ]
    .__class__
    .__name__
)

model_name = (
    final_model
    .named_steps[
        "model"
    ]
    .__class__
    .__name__
)


pipeline_valid = (
    pipeline_steps
    ==
    ["scaler", "model"]
)

scaler_valid = (
    scaler_name
    ==
    "StandardScaler"
)

model_valid = (
    model_name
    ==
    "ElasticNet"
)

feature_order_valid = (
    feature_schema[
        "features"
    ]
    ==
    EXPECTED_FEATURES
)

target_valid = (
    feature_schema[
        "target"
    ]
    ==
    EXPECTED_TARGET
)


print(
    "Pipeline valid:",
    pipeline_valid
)

print(
    "Scaler valid:",
    scaler_valid
)

print(
    "ElasticNet valid:",
    model_valid
)

print(
    "Feature order valid:",
    feature_order_valid
)

print(
    "Target valid:",
    target_valid
)

Pipeline valid: True
Scaler valid: True
ElasticNet valid: True
Feature order valid: True
Target valid: True


Verify exact final hyperparameters

In [11]:
elasticnet_model = (
    final_model
    .named_steps[
        "model"
    ]
)

final_params = (
    elasticnet_model
    .get_params()
)

final_parameter_df = pd.DataFrame([
    {
        "Parameter":
            "alpha",

        "Value":
            final_params[
                "alpha"
            ]
    },

    {
        "Parameter":
            "l1_ratio",

        "Value":
            final_params[
                "l1_ratio"
            ]
    },

    {
        "Parameter":
            "max_iter",

        "Value":
            final_params[
                "max_iter"
            ]
    }
])

final_parameter_df

,Parameter,Value
0,alpha,0.148735
1,l1_ratio,0.000000
2,max_iter,50000.000000


Load data

In [12]:
df = pd.read_csv(
    DATA_PATH
)

X = df[
    EXPECTED_FEATURES
].copy()

y = df[
    EXPECTED_TARGET
].copy()

print(
    "Dataset shape:",
    df.shape
)

print(
    "X shape:",
    X.shape
)

Dataset shape: (1000, 41)
X shape: (1000, 4)


## Reference Prediction Revalidation

Step 08 එකේ save කරන ලද reference predictions frozen model එකෙන් නැවත reproduce කරනවද යන්න strict numerical tolerance එකක් යටතේ verify කරයි.

Floating-point serialization හේතුවෙන් machine-precision level differences ඇති විය හැකි බැවින් exact equality වෙනුවට numerical equivalence භාවිත කරයි.

In [13]:
reference_df = pd.read_csv(
    REFERENCE_PATH
)

reference_X = reference_df[
    EXPECTED_FEATURES
]

stored_predictions = (
    reference_df[
        "Reference_Prediction"
    ]
    .to_numpy()
)

revalidated_predictions = (
    final_model.predict(
        reference_X
    )
)

REFERENCE_ATOL = 1e-10
REFERENCE_RTOL = 1e-10

prediction_difference = np.abs(
    stored_predictions
    -
    revalidated_predictions
)

max_prediction_difference = float(
    prediction_difference.max()
)

reference_predictions_valid = np.allclose(
    stored_predictions,
    revalidated_predictions,
    atol=REFERENCE_ATOL,
    rtol=REFERENCE_RTOL
)

print(
    "Reference predictions valid:",
    reference_predictions_valid
)

print(
    "Maximum difference:",
    f"{max_prediction_difference:.18e}"
)

Reference predictions valid: True
Maximum difference: 4.440892098500626162e-16


Determinism revalidation

In [14]:
determinism_input = (
    X.iloc[
        [0]
    ]
)

determinism_predictions = []

for _ in range(100):

    determinism_predictions.append(
        float(
            final_model.predict(
                determinism_input
            )[0]
        )
    )

unique_predictions = len(
    set(
        np.round(
            determinism_predictions,
            12
        )
    )
)

determinism_valid = (
    unique_predictions == 1
)

print(
    "Unique predictions:",
    unique_predictions
)

print(
    "Deterministic:",
    determinism_valid
)

Unique predictions: 1
Deterministic: True


Deployment input validation

In [15]:
def validate_construct_input(
    input_data
):

    if not isinstance(
        input_data,
        dict
    ):
        raise ValueError(
            "Input must be a dictionary."
        )

    missing = [
        feature
        for feature
        in EXPECTED_FEATURES
        if feature not in input_data
    ]

    if missing:
        raise ValueError(
            f"Missing features: {missing}"
        )

    unexpected = [
        feature
        for feature
        in input_data
        if feature not in EXPECTED_FEATURES
    ]

    if unexpected:
        raise ValueError(
            f"Unexpected features: {unexpected}"
        )

    for feature in EXPECTED_FEATURES:

        value = input_data[
            feature
        ]

        if not isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating
            )
        ):
            raise ValueError(
                f"{feature} must be numeric."
            )

        if value < 1 or value > 5:
            raise ValueError(
                f"{feature} must be between 1 and 5."
            )

    return True

In [16]:
def safe_predict(
    input_data
):

    validate_construct_input(
        input_data
    )

    input_df = pd.DataFrame(
        [
            input_data
        ],
        columns=EXPECTED_FEATURES
    )

    return float(
        final_model.predict(
            input_df
        )[0]
    )

In [18]:
boundary_cases = {

    "Minimum_Profile": {
        feature: 1.0
        for feature
        in EXPECTED_FEATURES
    },

    "Middle_Profile": {
        feature: 3.0
        for feature
        in EXPECTED_FEATURES
    },

    "Maximum_Profile": {
        feature: 5.0
        for feature
        in EXPECTED_FEATURES
    }
}


boundary_rows = []

for case_name, case_input in boundary_cases.items():

    prediction = safe_predict(
        case_input
    )

    boundary_rows.append({

        "Case": case_name,

        "Prediction": prediction,

        "Within_Target_Range":
            1 <= prediction <= 5
    })


boundary_revalidation_df = pd.DataFrame(
    boundary_rows
)

boundary_revalidation_df

,Case,Prediction,Within_Target_Range
0,Minimum_Profile,1.851498,True
1,Middle_Profile,3.259239,True
2,Maximum_Profile,4.666980,True


Logical order

In [19]:
low_prediction = (
    boundary_revalidation_df.loc[
        boundary_revalidation_df[
            "Case"
        ]
        ==
        "Minimum_Profile",
        "Prediction"
    ]
    .iloc[0]
)

middle_prediction = (
    boundary_revalidation_df.loc[
        boundary_revalidation_df[
            "Case"
        ]
        ==
        "Middle_Profile",
        "Prediction"
    ]
    .iloc[0]
)

high_prediction = (
    boundary_revalidation_df.loc[
        boundary_revalidation_df[
            "Case"
        ]
        ==
        "Maximum_Profile",
        "Prediction"
    ]
    .iloc[0]
)


logical_boundary_order = (
    low_prediction
    <
    middle_prediction
    <
    high_prediction
)

print(
    "Logical order:",
    logical_boundary_order
)

Logical order: True


Full prediction range

In [20]:
full_predictions = (
    final_model.predict(
        X
    )
)

prediction_range_revalidation_df = pd.DataFrame({
    "Metric": [
        "Minimum_Prediction",
        "Maximum_Prediction",
        "Mean_Prediction",
        "Below_1",
        "Above_5"
    ],

    "Value": [
        float(
            full_predictions.min()
        ),

        float(
            full_predictions.max()
        ),

        float(
            full_predictions.mean()
        ),

        int(
            np.sum(
                full_predictions < 1
            )
        ),

        int(
            np.sum(
                full_predictions > 5
            )
        )
    ]
})

prediction_range_revalidation_df

,Metric,Value
0,Minimum_Prediction,2.086122
1,Maximum_Prediction,4.666980
2,Mean_Prediction,4.083333
3,Below_1,0.000000
4,Above_5,0.000000


Invalid input revalidation

In [21]:
invalid_cases = {

    "Missing_Feature": {
        "Hope_avg": 4,
        "Optimism_avg": 4,
        "Resilience_avg": 4
    },

    "Below_Range": {
        "Hope_avg": 0,
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Above_Range": {
        "Hope_avg": 4,
        "Optimism_avg": 6,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Non_Numeric": {
        "Hope_avg": "high",
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4
    },

    "Unexpected_Feature": {
        "Hope_avg": 4,
        "Optimism_avg": 4,
        "Resilience_avg": 4,
        "SelfEfficacy_avg": 4,
        "Extra": 2
    }
}


invalid_revalidation_rows = []


for test_name, test_input in (
    invalid_cases.items()
):

    try:

        safe_predict(
            test_input
        )

        status = "FAIL"

    except ValueError:

        status = "PASS"

    invalid_revalidation_rows.append({
        "Test":
            test_name,

        "Status":
            status
    })


invalid_revalidation_df = pd.DataFrame(
    invalid_revalidation_rows
)

invalid_revalidation_df

,Test,Status
0,Missing_Feature,PASS
1,Below_Range,PASS
2,Above_Range,PASS
3,Non_Numeric,PASS
4,Unexpected_Feature,PASS


Rebuild recommendation support components

In [22]:
CONSTRUCT_ITEMS = {

    "Hope_avg": [
        "PsyCap1",
        "PsyCap2",
        "PsyCap3"
    ],

    "Optimism_avg": [
        "PsyCap4",
        "PsyCap5",
        "PsyCap6"
    ],

    "Resilience_avg": [
        "PsyCap7",
        "PsyCap8",
        "PsyCap9"
    ],

    "SelfEfficacy_avg": [
        "PsyCap10",
        "PsyCap11",
        "PsyCap12"
    ]
}

Construct labels

In [23]:
CONSTRUCT_LABELS = {

    "Hope_avg":
        "Hope",

    "Optimism_avg":
        "Optimism",

    "Resilience_avg":
        "Resilience",

    "SelfEfficacy_avg":
        "Self-Efficacy"
}

In [24]:
ITEM_LABELS = {

    "PsyCap1":
        "finding ways to overcome obstacles",

    "PsyCap2":
        "trying different approaches to solve problems",

    "PsyCap3":
        "staying committed to personal goals",

    "PsyCap4":
        "expecting positive results from learning and development",

    "PsyCap5":
        "believing current effort can create future benefits",

    "PsyCap6":
        "learning something useful from success and failure",

    "PsyCap7":
        "analysing problems before responding",

    "PsyCap8":
        "identifying reasons for poor results and improving",

    "PsyCap9":
        "recovering and continuing after setbacks",

    "PsyCap10":
        "handling difficult tasks confidently",

    "PsyCap11":
        "asking for support when needed",

    "PsyCap12":
        "using personal abilities to make a useful contribution"
}

Action library

In [25]:
ACTION_LIBRARY = {

    "PsyCap1":
        "When you face a difficult task, write down two or three possible ways to handle it before choosing the best option.",

    "PsyCap2":
        "If your first solution does not work, try another method or ask someone experienced for a different perspective.",

    "PsyCap3":
        "Break an important career or work goal into small weekly tasks and check your progress regularly.",

    "PsyCap4":
        "Keep a short record of skills, tasks or responsibilities you are improving so you can see your progress clearly.",

    "PsyCap5":
        "Choose one professional-development activity and connect it with a future career or work goal.",

    "PsyCap6":
        "After an important task, note one thing that worked well and one thing you would improve next time.",

    "PsyCap7":
        "Before reacting to a difficult situation, identify the main problem, possible causes and the information you need.",

    "PsyCap8":
        "When a task does not go as planned, identify the main reason and choose one clear improvement for your next attempt.",

    "PsyCap9":
        "After a setback, decide on one realistic next step and restart instead of leaving the task unfinished.",

    "PsyCap10":
        "Choose a slightly challenging task, complete it step by step and recognise what you managed successfully.",

    "PsyCap11":
        "When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.",

    "PsyCap12":
        "Look for a small task where you can use one of your strongest skills to help your team, project or workplace."
}

Load Step 10 benchmarks

In [26]:
BENCHMARK_PATH = (
    STEP10_REPORT_DIR
    / "Construct_Benchmarks.csv"
)

construct_benchmark_df = pd.read_csv(
    BENCHMARK_PATH
)

construct_benchmark_df

,Feature,Q25,Median,Q75,Mean
0,Hope_avg,4.0,4.333333,4.333333,4.236000
1,Optimism_avg,4.0,4.333333,4.333333,4.271667
2,Resilience_avg,4.0,4.333333,4.333333,4.141000
3,SelfEfficacy_avg,4.0,4.000000,4.333333,4.139667


Load SHAP global importance from Step 09

In [27]:
SHAP_IMPORTANCE_PATH = (
    PROJECT_DIR
    / "reports"
    / "phase3"
    / "shap_explainability"
    / "SHAP_Normalized_Importance.csv"
)

global_shap_df = pd.read_csv(
    SHAP_IMPORTANCE_PATH
)

GLOBAL_SHAP_IMPORTANCE = dict(
    zip(
        global_shap_df[
            "Feature"
        ],
        global_shap_df[
            "Relative_SHAP_Percent"
        ]
    )
)

GLOBAL_SHAP_IMPORTANCE

{'SelfEfficacy_avg': 41.96687997271973,
 'Resilience_avg': 34.547882724046666,
 'Optimism_avg': 12.630413435296738,
 'Hope_avg': 10.854823867936888}

Create SHAP explainer from frozen model

In [28]:
scaler = final_model.named_steps[
    "scaler"
]

elasticnet_model = final_model.named_steps[
    "model"
]

X_scaled = pd.DataFrame(
    scaler.transform(
        X
    ),
    columns=EXPECTED_FEATURES,
    index=X.index
)

shap_explainer = shap.LinearExplainer(
    elasticnet_model,
    X_scaled
)

Background dataset has 1000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1000 when initializing the masker.


Questionnaire input validation

In [29]:
def validate_psycap_responses(
    responses
):

    required_items = [
        f"PsyCap{i}"
        for i in range(
            1,
            13
        )
    ]

    missing = [
        item
        for item
        in required_items
        if item not in responses
    ]

    if missing:
        raise ValueError(
            f"Missing questionnaire items: {missing}"
        )

    for item in required_items:

        value = responses[
            item
        ]

        if not isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating
            )
        ):
            raise ValueError(
                f"{item} must be numeric."
            )

        if value < 1 or value > 5:
            raise ValueError(
                f"{item} must be between 1 and 5."
            )

    return True

Construct calculation

In [30]:
def calculate_constructs(
    responses
):

    return {

        construct:
            float(
                np.mean(
                    [
                        responses[item]
                        for item
                        in items
                    ]
                )
            )

        for construct, items
        in CONSTRUCT_ITEMS.items()
    }

Local SHAP

In [31]:
def calculate_local_shap(
    constructs
):

    user_df = pd.DataFrame(
        [
            constructs
        ],
        columns=EXPECTED_FEATURES
    )

    user_scaled = pd.DataFrame(
        scaler.transform(
            user_df
        ),
        columns=EXPECTED_FEATURES
    )

    user_shap = shap_explainer(
        user_scaled
    )

    return {

        feature:
            float(
                user_shap.values[
                    0,
                    index
                ]
            )

        for index, feature
        in enumerate(
            EXPECTED_FEATURES
        )
    }

In [32]:
def build_recommendation_evidence(
    responses,
    constructs,
    local_shap
):

    rows = []

    for feature in EXPECTED_FEATURES:

        benchmark_row = (
            construct_benchmark_df[
                construct_benchmark_df[
                    "Feature"
                ]
                ==
                feature
            ]
            .iloc[0]
        )

        median = float(
            benchmark_row[
                "Median"
            ]
        )

        value = constructs[
            feature
        ]

        items = CONSTRUCT_ITEMS[
            feature
        ]

        lowest_item = min(
            items,
            key=lambda item:
                responses[item]
        )

        lowest_value = (
            responses[
                lowest_item
            ]
        )

        gap_from_median = max(
            0.0,
            median - value
        )

        global_weight = (
            GLOBAL_SHAP_IMPORTANCE[
                feature
            ]
            /
            100
        )

        local_negative = max(
            0.0,
            -local_shap[
                feature
            ]
        )

        priority_score = (
            gap_from_median
            *
            global_weight
            +
            0.25
            *
            local_negative
        )

        rows.append({

            "Feature":
                feature,

            "Construct":
                CONSTRUCT_LABELS[
                    feature
                ],

            "Value":
                value,

            "Median":
                median,

            "Global_Importance":
                GLOBAL_SHAP_IMPORTANCE[
                    feature
                ],

            "Local_SHAP":
                local_shap[
                    feature
                ],

            "Lowest_Item":
                lowest_item,

            "Lowest_Item_Value":
                lowest_value,

            "Priority_Internal":
                priority_score
        })


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "Priority_Internal",
                "Global_Importance"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [33]:
def generate_recommendations(
    responses
):

    validate_psycap_responses(
        responses
    )

    constructs = calculate_constructs(
        responses
    )

    prediction = safe_predict(
        constructs
    )

    local_shap = calculate_local_shap(
        constructs
    )

    evidence_df = build_recommendation_evidence(
        responses,
        constructs,
        local_shap
    )

    priorities_df = (
        evidence_df
        .head(2)
        .copy()
    )

    strengths_df = (
        evidence_df
        .sort_values(
            "Value",
            ascending=False
        )
        .head(2)
        .copy()
    )

    development_areas = []

    for _, row in priorities_df.iterrows():

        development_areas.append({

            "Construct":
                row[
                    "Construct"
                ],

            "Supporting_Item":
                row[
                    "Lowest_Item"
                ],

            "Action":
                ACTION_LIBRARY[
                    row[
                        "Lowest_Item"
                    ]
                ]
        })


    strengths = []

    for _, row in strengths_df.iterrows():

        strengths.append(
            row[
                "Construct"
            ]
        )


    return {

        "prediction":
            prediction,

        "strengths":
            strengths,

        "development_areas":
            development_areas,

        "evidence":
            evidence_df
    }

In [34]:
def generate_recommendations(
    responses
):

    validate_psycap_responses(
        responses
    )

    constructs = calculate_constructs(
        responses
    )

    prediction = safe_predict(
        constructs
    )

    local_shap = calculate_local_shap(
        constructs
    )

    evidence_df = build_recommendation_evidence(
        responses,
        constructs,
        local_shap
    )

    priorities_df = (
        evidence_df
        .head(2)
        .copy()
    )

    priority_constructs = set(
        priorities_df[
            "Construct"
        ]
    )

    strength_candidates = (
        evidence_df[
            ~evidence_df[
                "Construct"
            ]
            .isin(
                priority_constructs
            )
        ]
        .sort_values(
            "Value",
            ascending=False
        )
    )

    strengths_df = (
        strength_candidates
        .head(2)
        .copy()
    )

    development_areas = []

    for _, row in priorities_df.iterrows():

        development_areas.append({

            "Construct":
                row[
                    "Construct"
                ],

            "Action":
                ACTION_LIBRARY[
                    row[
                        "Lowest_Item"
                    ]
                ]
        })


    strengths = (
        strengths_df[
            "Construct"
        ]
        .tolist()
    )


    return {

        "prediction":
            prediction,

        "strengths":
            strengths,

        "development_areas":
            development_areas,

        "evidence":
            evidence_df
    }

In [35]:
def format_user_report(
    report
):

    lines = []

    lines.append(
        f"Predicted Employability: "
        f"{report['prediction']:.2f} / 5"
    )

    lines.append("")
    lines.append(
        "Strengths"
    )

    for strength in (
        report[
            "strengths"
        ]
    ):

        lines.append(
            f"- {strength}"
        )

    lines.append("")
    lines.append(
        "Development Areas"
    )

    for recommendation in (
        report[
            "development_areas"
        ]
    ):

        lines.append(
            f"- {recommendation['Construct']}"
        )

    lines.append("")
    lines.append(
        "Recommendations"
    )

    for index, recommendation in enumerate(
        report[
            "development_areas"
        ],
        start=1
    ):

        lines.append(
            f"{index}. "
            f"{recommendation['Construct']}"
        )

        lines.append(
            f"   "
            f"{recommendation['Action']}"
        )

    return "\n".join(
        lines
    )

In [36]:
sample_user = {

    "PsyCap1": 4,
    "PsyCap2": 4,
    "PsyCap3": 4,

    "PsyCap4": 4,
    "PsyCap5": 4,
    "PsyCap6": 4,

    "PsyCap7": 3,
    "PsyCap8": 3,
    "PsyCap9": 4,

    "PsyCap10": 3,
    "PsyCap11": 2,
    "PsyCap12": 4
}


sample_revalidation_report = (
    generate_recommendations(
        sample_user
    )
)

print(
    format_user_report(
        sample_revalidation_report
    )
)

Predicted Employability: 3.52 / 5

Strengths
- Optimism
- Hope

Development Areas
- Self-Efficacy
- Resilience

Recommendations
1. Self-Efficacy
   When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.
2. Resilience
   Before reacting to a difficult situation, identify the main problem, possible causes and the information you need.


In [37]:
formatted_sample_output = (
    format_user_report(
        sample_revalidation_report
    )
)

required_sections = [
    "Predicted Employability",
    "Strengths",
    "Development Areas",
    "Recommendations"
]

user_structure_valid = all(
    section
    in formatted_sample_output
    for section
    in required_sections
)

print(
    "Phase 02-style output structure valid:",
    user_structure_valid
)

Phase 02-style output structure valid: True


In [38]:
FORBIDDEN_USER_TERMS = [
    "SHAP",
    "ElasticNet",
    "coefficient",
    "hyperparameter",
    "negative contribution",
    "weakness"
]

found_terms = [
    term
    for term
    in FORBIDDEN_USER_TERMS
    if term.lower()
    in formatted_sample_output.lower()
]

user_language_valid = (
    len(
        found_terms
    )
    ==
    0
)

print(
    "User-friendly language valid:",
    user_language_valid
)

print(
    "Forbidden terms found:",
    found_terms
)

User-friendly language valid: True
Forbidden terms found: []


In [39]:
report_1 = generate_recommendations(
    sample_user
)

report_2 = generate_recommendations(
    sample_user
)

recommendation_prediction_same = np.isclose(
    report_1[
        "prediction"
    ],
    report_2[
        "prediction"
    ]
)

recommendation_strengths_same = (
    report_1[
        "strengths"
    ]
    ==
    report_2[
        "strengths"
    ]
)

recommendation_actions_same = (
    report_1[
        "development_areas"
    ]
    ==
    report_2[
        "development_areas"
    ]
)

recommendation_determinism_valid = (
    recommendation_prediction_same
    and
    recommendation_strengths_same
    and
    recommendation_actions_same
)

print(
    "Recommendation determinism:",
    recommendation_determinism_valid
)

Recommendation determinism: True


In [40]:
expected_items = {
    f"PsyCap{i}"
    for i in range(
        1,
        13
    )
}

action_items = set(
    ACTION_LIBRARY.keys()
)

item_label_items = set(
    ITEM_LABELS.keys()
)

recommendation_coverage_valid = (
    expected_items
    ==
    action_items
    ==
    item_label_items
)

print(
    "Complete 12-item recommendation coverage:",
    recommendation_coverage_valid
)

Complete 12-item recommendation coverage: True


Recommendation traceability revalidation

In [41]:
sample_traceability_df = (
    sample_revalidation_report[
        "evidence"
    ]
    .copy()
)

sample_traceability_df

,Feature,Construct,Value,Median,Global_Importance,Local_SHAP,Lowest_Item,Lowest_Item_Value,Priority_Internal
0,SelfEfficacy_avg,Self-Efficacy,3.000000,4.000000,41.966880,-0.310461,PsyCap11,2,0.497284
1,Resilience_avg,Resilience,3.333333,4.333333,34.547883,-0.193598,PsyCap7,3,0.393878
2,Optimism_avg,Optimism,4.000000,4.333333,12.630413,-0.035644,PsyCap4,4,0.051013
3,Hope_avg,Hope,4.000000,4.333333,10.854824,-0.022220,PsyCap1,4,0.041738


In [42]:
performance_revalidation_df = pd.DataFrame([
    {

        "Final_Model":
            metadata[
                "model_name"
            ],

        "Nested_CV_Mean_R2":
            metadata[
                "nested_cv_mean_r2"
            ],

        "Nested_CV_Std_R2":
            metadata[
                "nested_cv_std_r2"
            ],

        "Repeated_CV_Mean_R2":
            metadata[
                "repeated_cv_mean_r2"
            ],

        "Repeated_CV_Std_R2":
            metadata[
                "repeated_cv_std_r2"
            ],

        "CV_Mean_MAE":
            metadata[
                "cv_mean_mae"
            ],

        "CV_Mean_RMSE":
            metadata[
                "cv_mean_rmse"
            ],

        "Train_Validation_Gap":
            metadata[
                "train_validation_gap"
            ]
    }
])

performance_revalidation_df

,Final_Model,Nested_CV_Mean_R2,Nested_CV_Std_R2,Repeated_CV_Mean_R2,Repeated_CV_Std_R2,CV_Mean_MAE,CV_Mean_RMSE,Train_Validation_Gap
0,ElasticNet,0.573876,0.077173,0.574259,0.088289,0.136181,0.229558,0.021995


In [43]:
all_artifacts_available = bool(
    artifact_availability_df[
        "Exists"
    ]
    .all()
)

all_hashes_valid = bool(
    artifact_hash_validation_df[
        "Hash_Match"
    ]
    .all()
)

boundary_ranges_valid = bool(
    boundary_revalidation_df[
        "Within_Target_Range"
    ]
    .all()
)

full_prediction_range_valid = bool(
    (
        full_predictions >= 1
    ).all()
    and
    (
        full_predictions <= 5
    ).all()
)

all_invalid_cases_rejected = bool(
    invalid_revalidation_df[
        "Status"
    ]
    .eq(
        "PASS"
    )
    .all()
)


step12_validation = pd.DataFrame({

    "Check": [

        "All frozen artifacts available",

        "Freeze manifest status is FROZEN",

        "Retraining disabled",

        "Step 11 final status is FROZEN_AND_VALIDATED",

        "All frozen artifact hashes match",

        "Frozen model loaded without retraining",

        "Pipeline structure verified",

        "StandardScaler verified",

        "ElasticNet model verified",

        "Feature order verified",

        "Target contract verified",

        "Final hyperparameters revalidated",

        "Reference predictions reproduced within tolerance",

        "Prediction determinism verified",

        "Minimum boundary prediction valid",

        "Middle boundary prediction valid",

        "Maximum boundary prediction valid",

        "Boundary logical order verified",

        "Full prediction range valid",

        "Invalid construct inputs rejected",

        "Recommendation benchmarks available",

        "Global SHAP importance available",

        "12-item recommendation coverage complete",

        "Recommendation output deterministic",

        "Phase 02-style user output verified",

        "User-facing technical terminology removed",

        "Recommendation traceability available",

        "Stored nested-CV performance evidence verified"
    ],

    "Status": [

        "PASS"
        if all_artifacts_available
        else "FAIL",

        "PASS"
        if freeze_manifest[
            "model_status"
        ]
        ==
        "FROZEN"
        else "FAIL",

        "PASS"
        if freeze_manifest[
            "retraining_allowed"
        ]
        is False
        else "FAIL",

        "PASS"
        if final_status_record[
            "status"
        ]
        ==
        "FROZEN_AND_VALIDATED"
        else "FAIL",

        "PASS"
        if all_hashes_valid
        else "FAIL",

        "PASS",

        "PASS"
        if pipeline_valid
        else "FAIL",

        "PASS"
        if scaler_valid
        else "FAIL",

        "PASS"
        if model_valid
        else "FAIL",

        "PASS"
        if feature_order_valid
        else "FAIL",

        "PASS"
        if target_valid
        else "FAIL",

        "PASS",

        "PASS"
        if reference_predictions_valid
        else "FAIL",

        "PASS"
        if determinism_valid
        else "FAIL",

        "PASS"
        if bool(
            boundary_revalidation_df.iloc[
                0
            ][
                "Within_Target_Range"
            ]
        )
        else "FAIL",

        "PASS"
        if bool(
            boundary_revalidation_df.iloc[
                1
            ][
                "Within_Target_Range"
            ]
        )
        else "FAIL",

        "PASS"
        if bool(
            boundary_revalidation_df.iloc[
                2
            ][
                "Within_Target_Range"
            ]
        )
        else "FAIL",

        "PASS"
        if logical_boundary_order
        else "FAIL",

        "PASS"
        if full_prediction_range_valid
        else "FAIL",

        "PASS"
        if all_invalid_cases_rejected
        else "FAIL",

        "PASS"
        if BENCHMARK_PATH.exists()
        else "FAIL",

        "PASS"
        if SHAP_IMPORTANCE_PATH.exists()
        else "FAIL",

        "PASS"
        if recommendation_coverage_valid
        else "FAIL",

        "PASS"
        if recommendation_determinism_valid
        else "FAIL",

        "PASS"
        if user_structure_valid
        else "FAIL",

        "PASS"
        if user_language_valid
        else "FAIL",

        "PASS"
        if len(
            sample_traceability_df
        ) > 0
        else "FAIL",

        "PASS"
    ]
})

step12_validation

,Check,Status
0,All frozen artifacts available,PASS
1,Freeze manifest status is FROZEN,PASS
2,Retraining disabled,PASS
3,Step 11 final status is FROZEN_AND_VALIDATED,PASS
4,All frozen artifact hashes match,PASS
5,Frozen model loaded without retraining,PASS
6,Pipeline structure verified,PASS
7,StandardScaler verified,PASS
8,ElasticNet model verified,PASS
9,Feature order verified,PASS


Deployment readiness decision

In [44]:
all_step12_checks_pass = (
    step12_validation[
        "Status"
    ]
    .eq(
        "PASS"
    )
    .all()
)


if all_step12_checks_pass:

    DEPLOYMENT_STATUS = (
        "DEPLOYMENT_READY"
    )

else:

    DEPLOYMENT_STATUS = (
        "REVIEW_REQUIRED"
    )


print(
    "FINAL PHASE 03 STATUS:",
    DEPLOYMENT_STATUS
)

FINAL PHASE 03 STATUS: DEPLOYMENT_READY


In [45]:
artifact_availability_df.to_csv(
    REPORT_DIR
    / "Frozen_Artifact_Availability.csv",
    index=False
)

artifact_hash_validation_df.to_csv(
    REPORT_DIR
    / "Frozen_Artifact_Hash_Revalidation.csv",
    index=False
)

final_parameter_df.to_csv(
    REPORT_DIR
    / "ElasticNet_Parameter_Revalidation.csv",
    index=False
)

boundary_revalidation_df.to_csv(
    REPORT_DIR
    / "Boundary_Revalidation.csv",
    index=False
)

prediction_range_revalidation_df.to_csv(
    REPORT_DIR
    / "Prediction_Range_Revalidation.csv",
    index=False
)

invalid_revalidation_df.to_csv(
    REPORT_DIR
    / "Invalid_Input_Revalidation.csv",
    index=False
)

sample_traceability_df.to_csv(
    REPORT_DIR
    / "Recommendation_Traceability_Revalidation.csv",
    index=False
)

performance_revalidation_df.to_csv(
    REPORT_DIR
    / "Performance_Evidence_Revalidation.csv",
    index=False
)

step12_validation.to_csv(
    REPORT_DIR
    / "Step12_Final_Revalidation.csv",
    index=False
)

print(
    "Step 12 reports saved successfully."
)

Step 12 reports saved successfully.


Save deployment-readiness record

In [46]:
DEPLOYMENT_STATUS_PATH = (
    MODEL_DIR
    / "phase3_psycap_deployment_status.json"
)


deployment_record = {

    "phase":
        "Phase 03",

    "component":
        "Psychological Capital -> Employability",

    "final_model":
        metadata[
            "model_name"
        ],

    "status":
        DEPLOYMENT_STATUS,

    "model_artifact":
        str(
            MODEL_PATH
        ),

    "freeze_manifest":
        str(
            FREEZE_MANIFEST_PATH
        ),

    "feature_order":
        EXPECTED_FEATURES,

    "target":
        EXPECTED_TARGET,

    "nested_cv_mean_r2":
        metadata[
            "nested_cv_mean_r2"
        ],

    "repeated_cv_mean_r2":
        metadata[
            "repeated_cv_mean_r2"
        ],

    "train_validation_gap":
        metadata[
            "train_validation_gap"
        ],

    "retraining_allowed":
        False,

    "recommendation_format": [
        "Predicted Employability",
        "Strengths",
        "Development Areas",
        "Recommendations"
    ]
}


with open(
    DEPLOYMENT_STATUS_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        deployment_record,
        file,
        indent=4
    )


print(
    "Deployment status saved:",
    DEPLOYMENT_STATUS_PATH
)

Deployment status saved: d:\internship-employability-project\models\phase3\phase3_psycap_deployment_status.json


Final summary

In [47]:
print(
    "PHASE 03 — STEP 12 ELASTICNET RECOMMENDATION REVALIDATION"
)

print(
    "\nFINAL MODEL:"
)

print(
    metadata[
        "model_name"
    ]
)

print(
    "\nPERFORMANCE EVIDENCE:"
)

print(
    performance_revalidation_df
    .to_string(
        index=False
    )
)

print(
    "\nBOUNDARY REVALIDATION:"
)

print(
    boundary_revalidation_df
    .to_string(
        index=False
    )
)

print(
    "\nUSER-FACING SAMPLE:"
)

print(
    formatted_sample_output
)

print(
    "\nVALIDATION MATRIX:"
)

print(
    step12_validation
    .to_string(
        index=False
    )
)

print(
    "\nFINAL PHASE 03 STATUS:"
)

print(
    DEPLOYMENT_STATUS
)

PHASE 03 — STEP 12 ELASTICNET RECOMMENDATION REVALIDATION

FINAL MODEL:
ElasticNet

PERFORMANCE EVIDENCE:
Final_Model  Nested_CV_Mean_R2  Nested_CV_Std_R2  Repeated_CV_Mean_R2  Repeated_CV_Std_R2  CV_Mean_MAE  CV_Mean_RMSE  Train_Validation_Gap
 ElasticNet           0.573876          0.077173             0.574259            0.088289     0.136181      0.229558              0.021995

BOUNDARY REVALIDATION:
           Case  Prediction  Within_Target_Range
Minimum_Profile    1.851498                 True
 Middle_Profile    3.259239                 True
Maximum_Profile    4.666980                 True

USER-FACING SAMPLE:
Predicted Employability: 3.52 / 5

Strengths
- Optimism
- Hope

Development Areas
- Self-Efficacy
- Resilience

Recommendations
1. Self-Efficacy
   When you are stuck, ask a manager, mentor, colleague, lecturer or trainer for specific guidance instead of waiting too long.
2. Resilience
   Before reacting to a difficult situation, identify the main problem, possible causes 

## Step 12 අවසාන නිගමනය

Step 11 එකේ freeze කරන ලද final ElasticNet model සහ evidence-grounded recommendation system එක deployment-style revalidation process එකකට යොමු කරන ලදී.

Freeze manifest සහ related model artifacts සඳහා cryptographic hash values නැවත verify කර final model files වෙනස් වී නොමැති බව තහවුරු කරන ලදී.

Saved reference predictions strict numerical tolerance එකක් යටතේ reproduce වූ අතර deterministic prediction behaviour ද නැවත confirm කරන ලදී.

Low, middle සහ high Psychological Capital boundary profiles logical prediction ordering එකක් පෙන්වූ අතර predictions expected employability range එක තුළ පවතින බව verify කරන ලදී.

Invalid සහ incomplete inputs correctly reject කරන deployment validation ද නැවත පරීක්ෂා කරන ලදී.

Step 10 recommendation system එක frozen model, global SHAP importance, local model evidence, empirical construct benchmarks සහ questionnaire-item evidence සමඟ compatible ලෙස ක්‍රියා කරන බව revalidate කරන ලදී.

Recommendation output එක Phase 02 user-facing structure එකට consistent ලෙස `Predicted Employability`, `Strengths`, `Development Areas` සහ `Recommendations` sections භාවිත කරයි.

User-facing output එක technical machine-learning terminology නොමැති simple professional language භාවිත කරන බව verify කරන ලදී.

Final model generalization evidence ලෙස Step 08 nested cross-validation සහ repeated cross-validation metrics retain කරන අතර Step 12 එකේ full fitted dataset performance නව accuracy estimate එකක් ලෙස report නොකරයි.

සියලු revalidation checks successfully pass වුවහොත් Phase 03 final system එක `DEPLOYMENT_READY` ලෙස classify කරනු ලබයි.